In [1]:
# ============================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 1 — CONFIGURATION + TEACHER VERIFICATION
# ============================================================

from pathlib import Path
import torch
from ultralytics import YOLO
import yaml


# ============================================================
# 1. DATASET PATHS
# ============================================================

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO\dataset.yaml"
)

TRAIN_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\train"
)

VAL_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\val"
)

TEST_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\test"
)

TRAIN_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\train"
)

VAL_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\val"
)

TEST_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\test"
)


# ============================================================
# 2. YOLOv8n TEACHER / PARENT MODEL
# ============================================================

TEACHER_PATH = Path(
    r"G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt"
)


# ============================================================
# 3. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\knowledge_distillation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. STUDENT CONFIGURATION
# ============================================================

STUDENT_NAME = "EcoBotX_Tiny_KD"

NUM_CLASSES = 4

CLASS_NAMES = [
    "BOTTLE",
    "CAN",
    "PAPER",
    "WRAPPER"
]


# ============================================================
# 5. TRAINING CONFIGURATION
# ============================================================

IMAGE_SIZE = 640

BATCH_SIZE = 8

EPOCHS = 100

LEARNING_RATE = 0.001

WEIGHT_DECAY = 0.0005

NUM_WORKERS = 4

SEED = 42

DEVICE = 0 if torch.cuda.is_available() else "cpu"


# ============================================================
# 6. KNOWLEDGE DISTILLATION CONFIGURATION
# ============================================================

KD_TEMPERATURE = 4.0

KD_CLASS_WEIGHT = 0.5

KD_BOX_WEIGHT = 0.5

KD_FEATURE_WEIGHT = 0.0


# ============================================================
# 7. PRINT ENVIRONMENT
# ============================================================

print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 1 — CONFIGURATION + TEACHER VERIFICATION")
print("=" * 70)

print()

print("PyTorch version :", torch.__version__)

print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory      :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )

print()


# ============================================================
# 8. PATH VALIDATION
# ============================================================

print("=" * 70)
print("PATH VALIDATION")
print("=" * 70)

paths_to_check = {

    "Dataset YAML": DATASET_YAML,

    "Train images": TRAIN_IMAGES,

    "Validation images": VAL_IMAGES,

    "Test images": TEST_IMAGES,

    "Train labels": TRAIN_LABELS,

    "Validation labels": VAL_LABELS,

    "Test labels": TEST_LABELS,

    "Teacher best.pt": TEACHER_PATH,
}


all_paths_valid = True


for name, path in paths_to_check.items():

    if path.exists():

        print(f"[OK] {name}")
        print(f"     {path}")

    else:

        print(f"[ERROR] {name} NOT FOUND")
        print(f"        {path}")

        all_paths_valid = False


print()


if not all_paths_valid:

    raise FileNotFoundError(
        "\nOne or more required paths are missing.\n"
        "Fix the paths before continuing to Section 2."
    )


# ============================================================
# 9. LOAD DATASET YAML
# ============================================================

print("=" * 70)
print("DATASET YAML VERIFICATION")
print("=" * 70)

with open(
    DATASET_YAML,
    "r",
    encoding="utf-8"
) as f:

    dataset_config = yaml.safe_load(f)


yaml_names = dataset_config.get(
    "names",
    []
)


print("Dataset classes:")

if isinstance(yaml_names, dict):

    for idx, name in yaml_names.items():

        print(
            f"  {idx}: {name}"
        )

else:

    for idx, name in enumerate(yaml_names):

        print(
            f"  {idx}: {name}"
        )


print()


# ============================================================
# 10. VERIFY CLASS COUNT
# ============================================================

yaml_num_classes = len(yaml_names)


if yaml_num_classes != NUM_CLASSES:

    raise ValueError(
        f"Dataset has {yaml_num_classes} classes, "
        f"but expected {NUM_CLASSES}."
    )


print(
    f"[OK] Dataset contains {NUM_CLASSES} classes."
)

print()


# ============================================================
# 11. LOAD YOLOv8n TEACHER
# ============================================================

print("=" * 70)
print("LOADING YOLOv8n TEACHER")
print("=" * 70)

print()

print(
    "Teacher checkpoint:"
)

print(
    TEACHER_PATH
)

print()


teacher = YOLO(
    str(TEACHER_PATH)
)


print(
    "[OK] YOLOv8n teacher loaded successfully."
)

print()


# ============================================================
# 12. TEACHER CLASS VERIFICATION
# ============================================================

print("=" * 70)
print("TEACHER CLASS VERIFICATION")
print("=" * 70)

print()

print("Teacher classes:")

for idx, name in teacher.names.items():

    print(
        f"  {idx}: {name}"
    )


print()


teacher_class_names = [
    teacher.names[i]
    for i in sorted(teacher.names.keys())
]


expected_class_names = CLASS_NAMES


if teacher_class_names == expected_class_names:

    print(
        "[OK] Teacher classes match dataset classes."
    )

else:

    print(
        "[ERROR] Teacher classes DO NOT match dataset."
    )

    print()

    print(
        "Expected:"
    )

    print(
        expected_class_names
    )

    print()

    print(
        "Teacher:"
    )

    print(
        teacher_class_names
    )

    raise ValueError(
        "Teacher and dataset class ordering do not match."
    )


print()


# ============================================================
# 13. TEACHER MODEL INFORMATION
# ============================================================

print("=" * 70)
print("TEACHER MODEL INFORMATION")
print("=" * 70)

print()

print(
    "Teacher path:"
)

print(
    TEACHER_PATH
)

print()

try:

    teacher.info(
        verbose=True
    )

except Exception as e:

    print(
        "[WARNING] Could not display detailed model info."
    )

    print(
        "Reason:",
        e
    )


print()


# ============================================================
# 14. VERIFY TEACHER WEIGHTS ARE AVAILABLE
# ============================================================

teacher_checkpoint_size_mb = (
    TEACHER_PATH.stat().st_size
    / (1024 ** 2)
)


print("=" * 70)
print("TEACHER CHECKPOINT")
print("=" * 70)

print()

print(
    f"Checkpoint size : "
    f"{teacher_checkpoint_size_mb:.2f} MB"
)

print(
    "[OK] Teacher checkpoint exists."
)

print()


# ============================================================
# 15. FINAL CONFIGURATION
# ============================================================

print("=" * 70)
print("FINAL KD CONFIGURATION")
print("=" * 70)

print()

print("TEACHER")
print("-" * 70)

print(
    "YOLOv8n:"
)

print(
    TEACHER_PATH
)

print()

print("STUDENT")
print("-" * 70)

print(
    "Student name:",
    STUDENT_NAME
)

print()

print("DATASET")
print("-" * 70)

print(
    "YAML:",
    DATASET_YAML
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Class names:",
    CLASS_NAMES
)

print()

print("TRAINING")
print("-" * 70)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Workers:",
    NUM_WORKERS
)

print(
    "Device:",
    DEVICE
)

print()

print("KNOWLEDGE DISTILLATION")
print("-" * 70)

print(
    "Temperature:",
    KD_TEMPERATURE
)

print(
    "Class KD weight:",
    KD_CLASS_WEIGHT
)

print(
    "Box KD weight:",
    KD_BOX_WEIGHT
)

print(
    "Feature KD weight:",
    KD_FEATURE_WEIGHT
)

print()

print("OUTPUT")
print("-" * 70)

print(
    OUTPUT_DIR
)

print()

print("=" * 70)
print("[OK] SECTION 1 COMPLETED")
print("=" * 70)

print()

print(
    "Ready for SECTION 2 — EcoBotX lightweight student architecture."
)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 1 — CONFIGURATION + TEACHER VERIFICATION

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory      : 4.0 GB

PATH VALIDATION
[OK] Dataset YAML
     G:\EcoBotX_YOLO\dataset.yaml
[OK] Train images
     G:\EcoBotX_YOLO\images\train
[OK] Validation images
     G:\EcoBotX_YOLO\images\val
[OK] Test images
     G:\EcoBotX_YOLO\images\test
[OK] Train labels
     G:\EcoBotX_YOLO\labels\train
[OK] Validation labels
     G:\EcoBotX_YOLO\labels\val
[OK] Test labels
     G:\EcoBotX_YOLO\labels\test
[OK] Teacher best.pt
     G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt

DATASET YAML VERIFICATION
Dataset classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

[OK] Dataset contains 4 classes.

LOADING YOLOv8n TEACHER

Teacher checkpoint:
G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt

[OK] YOLOv8n teacher loaded successfully.

TEACHER CLASS VERIFICATION

Teacher classes:
  0: 

In [2]:
# ============================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 2 — LIGHTWEIGHT STUDENT ARCHITECTURE
# CLEAN / KD-COMPATIBLE VERSION
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path


# ============================================================
# 1. CONFIGURATION
# ============================================================

NUM_CLASSES = 4

CLASS_NAMES = [
    "BOTTLE",
    "CAN",
    "PAPER",
    "WRAPPER"
]

IMAGE_SIZE = 640

STUDENT_NAME = "EcoBotX_Tiny_KD"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 2 — STUDENT ARCHITECTURE")
print("=" * 70)

print(f"Device : {DEVICE}")


# ============================================================
# 2. GHOST CONV
# ============================================================

class GhostConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=1,
        stride=1,
        activation=True
    ):

        super().__init__()

        padding = kernel_size // 2

        # Split output channels into primary + cheap features
        primary_channels = (out_channels + 1) // 2
        cheap_channels = out_channels - primary_channels

        self.primary = nn.Sequential(
            nn.Conv2d(
                in_channels,
                primary_channels,
                kernel_size,
                stride,
                padding,
                bias=False
            ),
            nn.BatchNorm2d(primary_channels),
            nn.ReLU6(inplace=True)
        )

        self.cheap = None

        if cheap_channels > 0:

            self.cheap = nn.Sequential(
                nn.Conv2d(
                    primary_channels,
                    cheap_channels,
                    kernel_size=3,
                    stride=1,
                    padding=1,
                    groups=primary_channels
                    if cheap_channels >= primary_channels
                    else 1,
                    bias=False
                ),
                nn.BatchNorm2d(cheap_channels),
                nn.ReLU6(inplace=True)
            )

        self.out_channels = out_channels

    def forward(self, x):

        y = self.primary(x)

        if self.cheap is None:
            return y

        z = self.cheap(y)

        return torch.cat(
            [y, z],
            dim=1
        )


# ============================================================
# 3. DEPTHWISE SEPARABLE CONV
# ============================================================

class DWConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()

        self.depthwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(in_channels),

            nn.ReLU6(inplace=True)
        )

        self.pointwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU6(inplace=True)
        )

    def forward(self, x):

        x = self.depthwise(x)

        x = self.pointwise(x)

        return x


# ============================================================
# 4. COORDINATE ATTENTION
# ============================================================

class CoordinateAttention(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):

        super().__init__()

        reduced_channels = max(
            8,
            channels // reduction
        )

        self.conv1 = nn.Conv2d(
            channels,
            reduced_channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(
            reduced_channels
        )

        self.act = nn.ReLU6(
            inplace=True
        )

        self.conv_h = nn.Conv2d(
            reduced_channels,
            channels,
            kernel_size=1
        )

        self.conv_w = nn.Conv2d(
            reduced_channels,
            channels,
            kernel_size=1
        )

    def forward(self, x):

        identity = x

        b, c, h, w = x.shape

        # Height-wise pooling
        x_h = x.mean(
            dim=3,
            keepdim=True
        )

        # Width-wise pooling
        x_w = x.mean(
            dim=2,
            keepdim=True
        )

        x_w = x_w.permute(
            0,
            1,
            3,
            2
        )

        y = torch.cat(
            [x_h, x_w],
            dim=2
        )

        y = self.conv1(y)

        y = self.bn1(y)

        y = self.act(y)

        y_h, y_w = torch.split(
            y,
            [h, w],
            dim=2
        )

        y_w = y_w.permute(
            0,
            1,
            3,
            2
        )

        attention_h = torch.sigmoid(
            self.conv_h(y_h)
        )

        attention_w = torch.sigmoid(
            self.conv_w(y_w)
        )

        return (
            identity
            * attention_h
            * attention_w
        )


# ============================================================
# 5. LIGHTWEIGHT RESIDUAL BLOCK
# ============================================================

class EcoBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        attention=False
    ):

        super().__init__()

        self.ghost = GhostConv(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1
        )

        self.dwconv = DWConv(
            out_channels,
            out_channels,
            stride=stride
        )

        if attention:

            self.attention = CoordinateAttention(
                out_channels
            )

        else:

            self.attention = nn.Identity()

        if (
            stride != 1
            or in_channels != out_channels
        ):

            self.shortcut = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(
                    out_channels
                )
            )

        else:

            self.shortcut = nn.Identity()

        self.act = nn.ReLU6(
            inplace=True
        )

    def forward(self, x):

        identity = self.shortcut(x)

        y = self.ghost(x)

        y = self.dwconv(y)

        y = self.attention(y)

        y = y + identity

        y = self.act(y)

        return y


# ============================================================
# 6. ECOBOT-X BACKBONE
#
# P3 = 64 channels @ stride 8
# P4 = 96 channels @ stride 16
# P5 = 128 channels @ stride 32
# ============================================================

class EcoBotXBackbone(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Stem
        # 640 -> 320
        # ----------------------------------------------------

        self.stem = GhostConv(
            3,
            32,
            kernel_size=3,
            stride=2
        )

        # ----------------------------------------------------
        # Stage 1
        # 320 -> 160
        # ----------------------------------------------------

        self.stage1 = EcoBlock(
            32,
            48,
            stride=2,
            attention=False
        )

        # ----------------------------------------------------
        # Stage 2
        # 160 -> 80
        # P3
        # ----------------------------------------------------

        self.stage2 = EcoBlock(
            48,
            64,
            stride=2,
            attention=False
        )

        self.stage2_refine = EcoBlock(
            64,
            64,
            stride=1,
            attention=False
        )

        # ----------------------------------------------------
        # Stage 3
        # 80 -> 40
        # P4
        # ----------------------------------------------------

        self.stage3 = EcoBlock(
            64,
            96,
            stride=2,
            attention=True
        )

        self.stage3_refine = EcoBlock(
            96,
            96,
            stride=1,
            attention=True
        )

        # ----------------------------------------------------
        # Stage 4
        # 40 -> 20
        # P5
        # ----------------------------------------------------

        self.stage4 = EcoBlock(
            96,
            128,
            stride=2,
            attention=True
        )

        self.stage4_refine = EcoBlock(
            128,
            128,
            stride=1,
            attention=True
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.stage1(x)

        # -------------------------
        # P3
        # -------------------------

        x = self.stage2(x)

        p3 = self.stage2_refine(x)

        # -------------------------
        # P4
        # -------------------------

        x = self.stage3(p3)

        p4 = self.stage3_refine(x)

        # -------------------------
        # P5
        # -------------------------

        x = self.stage4(p4)

        p5 = self.stage4_refine(x)

        return p3, p4, p5


# ============================================================
# 7. LIGHTWEIGHT GHOST-PAN
# ============================================================

class GhostPAN(nn.Module):

    def __init__(self):

        super().__init__()

        # ====================================================
        # TOP-DOWN P5 -> P4
        # ====================================================

        self.p5_to_p4 = GhostConv(
            128,
            96,
            kernel_size=1,
            stride=1
        )

        self.p4_fuse = GhostConv(
            192,
            96,
            kernel_size=1,
            stride=1
        )

        # ====================================================
        # TOP-DOWN P4 -> P3
        # ====================================================

        self.p4_to_p3 = GhostConv(
            96,
            64,
            kernel_size=1,
            stride=1
        )

        self.p3_fuse = GhostConv(
            128,
            64,
            kernel_size=1,
            stride=1
        )

        # ====================================================
        # BOTTOM-UP P3 -> P4
        # ====================================================

        self.p3_down = DWConv(
            64,
            96,
            stride=2
        )

        self.p4_pan = GhostConv(
            192,
            96,
            kernel_size=1,
            stride=1
        )

        # ====================================================
        # BOTTOM-UP P4 -> P5
        # ====================================================

        self.p4_down = DWConv(
            96,
            128,
            stride=2
        )

        self.p5_pan = GhostConv(
            256,
            128,
            kernel_size=1,
            stride=1
        )

    def forward(
        self,
        p3,
        p4,
        p5
    ):

        # ====================================================
        # TOP-DOWN
        # ====================================================

        p5_td = self.p5_to_p4(p5)

        p5_td = F.interpolate(
            p5_td,
            size=p4.shape[-2:],
            mode="nearest"
        )

        p4_td = torch.cat(
            [p4, p5_td],
            dim=1
        )

        p4_td = self.p4_fuse(
            p4_td
        )

        p4_up = self.p4_to_p3(
            p4_td
        )

        p4_up = F.interpolate(
            p4_up,
            size=p3.shape[-2:],
            mode="nearest"
        )

        p3_td = torch.cat(
            [p3, p4_up],
            dim=1
        )

        p3_out = self.p3_fuse(
            p3_td
        )

        # ====================================================
        # BOTTOM-UP
        # ====================================================

        p3_down = self.p3_down(
            p3_out
        )

        p4_out = torch.cat(
            [p3_down, p4_td],
            dim=1
        )

        p4_out = self.p4_pan(
            p4_out
        )

        p4_down = self.p4_down(
            p4_out
        )

        p5_out = torch.cat(
            [p4_down, p5],
            dim=1
        )

        p5_out = self.p5_pan(
            p5_out
        )

        return (
            p3_out,
            p4_out,
            p5_out
        )


# ============================================================
# 8. STUDENT FEATURE OUTPUT
#
# IMPORTANT:
# This section only produces feature maps.
#
# Detection decoding / YOLO-compatible prediction will be
# handled in later sections.
# ============================================================

class EcoBotXStudent(nn.Module):

    def __init__(
        self,
        num_classes=NUM_CLASSES
    ):

        super().__init__()

        self.num_classes = num_classes

        self.backbone = EcoBotXBackbone()

        self.neck = GhostPAN()

    def forward(self, x):

        # Backbone
        p3, p4, p5 = self.backbone(x)

        # Neck
        p3, p4, p5 = self.neck(
            p3,
            p4,
            p5
        )

        return {
            "P3": p3,
            "P4": p4,
            "P5": p5,

            "features": [
                p3,
                p4,
                p5
            ]
        }


# ============================================================
# 9. CREATE STUDENT
# ============================================================

print()
print("=" * 70)
print("CREATING ECOBOT-X STUDENT")
print("=" * 70)

student = EcoBotXStudent(
    num_classes=NUM_CLASSES
).to(DEVICE)


# ============================================================
# 10. PARAMETER COUNT
# ============================================================

total_parameters = sum(
    p.numel()
    for p in student.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)

parameter_size_mb = (
    total_parameters * 4
) / (
    1024 ** 2
)


# ============================================================
# 11. FORWARD PASS TEST
# ============================================================

print()
print("=" * 70)
print("FORWARD PASS TEST")
print("=" * 70)

student.eval()

dummy_input = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE
)

with torch.no_grad():

    output = student(
        dummy_input
    )


# ============================================================
# 12. VERIFY FEATURE SHAPES
# ============================================================

print()
print("Feature output shapes:")

for name in ["P3", "P4", "P5"]:

    feature = output[name]

    print(
        f"{name}: "
        f"{tuple(feature.shape)}"
    )


# ============================================================
# 13. STRICT ARCHITECTURE CHECK
# ============================================================

expected_shapes = {

    "P3": (
        1,
        64,
        80,
        80
    ),

    "P4": (
        1,
        96,
        40,
        40
    ),

    "P5": (
        1,
        128,
        20,
        20
    )
}


print()
print("=" * 70)
print("ARCHITECTURE VALIDATION")
print("=" * 70)

architecture_valid = True

for name, expected in expected_shapes.items():

    actual = tuple(
        output[name].shape
    )

    if actual == expected:

        print(
            f"[OK] {name}: "
            f"{actual}"
        )

    else:

        architecture_valid = False

        print(
            f"[ERROR] {name}: "
            f"expected {expected}, "
            f"got {actual}"
        )


# ============================================================
# 14. FINAL MODEL INFORMATION
# ============================================================

print()
print("=" * 70)
print("STUDENT MODEL INFORMATION")
print("=" * 70)

print(
    f"Architecture       : {STUDENT_NAME}"
)

print(
    "Backbone            : "
    "Ghost + DWConv + Coordinate Attention"
)

print(
    "Neck                : "
    "Lightweight Ghost-PAN"
)

print(
    "Detection levels    : P3 / P4 / P5"
)

print(
    "P3 channels         : 64"
)

print(
    "P4 channels         : 96"
)

print(
    "P5 channels         : 128"
)

print(
    f"Input size          : "
    f"{IMAGE_SIZE}x{IMAGE_SIZE}"
)

print(
    f"Total parameters    : "
    f"{total_parameters:,}"
)

print(
    f"Parameters (M)      : "
    f"{total_parameters / 1e6:.3f}"
)

print(
    f"FP32 size           : "
    f"{parameter_size_mb:.2f} MB"
)


# ============================================================
# 15. FINAL RESULT
# ============================================================

print()

if architecture_valid:

    print(
        "[OK] STUDENT ARCHITECTURE "
        "VALIDATED SUCCESSFULLY."
    )

    print()
    print(
        "P3 = [1, 64, 80, 80]"
    )

    print(
        "P4 = [1, 96, 40, 40]"
    )

    print(
        "P5 = [1, 128, 20, 20]"
    )

    print()
    print(
        "Ready for SECTION 3 — "
        "Teacher/Student feature alignment."
    )

else:

    raise RuntimeError(
        "Student architecture validation failed."
    )

print("=" * 70)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 2 — STUDENT ARCHITECTURE
Device : cuda

CREATING ECOBOT-X STUDENT

FORWARD PASS TEST

Feature output shapes:
P3: (1, 64, 80, 80)
P4: (1, 96, 40, 40)
P5: (1, 128, 20, 20)

ARCHITECTURE VALIDATION
[OK] P3: (1, 64, 80, 80)
[OK] P4: (1, 96, 40, 40)
[OK] P5: (1, 128, 20, 20)

STUDENT MODEL INFORMATION
Architecture       : EcoBotX_Tiny_KD
Backbone            : Ghost + DWConv + Coordinate Attention
Neck                : Lightweight Ghost-PAN
Detection levels    : P3 / P4 / P5
P3 channels         : 64
P4 channels         : 96
P5 channels         : 128
Input size          : 640x640
Total parameters    : 208,600
Parameters (M)      : 0.209
FP32 size           : 0.80 MB

[OK] STUDENT ARCHITECTURE VALIDATED SUCCESSFULLY.

P3 = [1, 64, 80, 80]
P4 = [1, 96, 40, 40]
P5 = [1, 128, 20, 20]

Ready for SECTION 3 — Teacher/Student feature alignment.


In [3]:
# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 3 — TEACHER / STUDENT FEATURE ALIGNMENT
# ======================================================================

import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 3 — TEACHER / STUDENT FEATURE ALIGNMENT")
print("=" * 70)

# ======================================================================
# CONFIGURATION
# ======================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEACHER_PATH = Path(
    r"G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt"
)

INPUT_SIZE = 640

NUM_CLASSES = 4

# Student feature channels from SECTION 2
STUDENT_CHANNELS = {
    "P3": 64,
    "P4": 96,
    "P5": 128,
}

# Expected spatial resolutions for 640x640 input
EXPECTED_RESOLUTIONS = {
    "P3": (80, 80),
    "P4": (40, 40),
    "P5": (20, 20),
}

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\knowledge_distillation"
    r"\section3_feature_alignment"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print()
print("Device       :", DEVICE)
print("Teacher      :", TEACHER_PATH)
print("Input size   :", f"{INPUT_SIZE}x{INPUT_SIZE}")
print("Student P3   :", STUDENT_CHANNELS["P3"])
print("Student P4   :", STUDENT_CHANNELS["P4"])
print("Student P5   :", STUDENT_CHANNELS["P5"])


# ======================================================================
# CHECK TEACHER
# ======================================================================

print()
print("=" * 70)
print("LOADING YOLOv8n TEACHER")
print("=" * 70)

if not TEACHER_PATH.exists():
    raise FileNotFoundError(
        f"Teacher model not found:\n{TEACHER_PATH}"
    )

from ultralytics import YOLO

teacher_wrapper = YOLO(str(TEACHER_PATH))

# Ultralytics YOLO model
teacher = teacher_wrapper.model

teacher = teacher.to(DEVICE)
teacher.eval()

# Freeze teacher
for param in teacher.parameters():
    param.requires_grad = False

print("[OK] Teacher loaded")
print("[OK] Teacher frozen")
print("Teacher type:", type(teacher).__name__)


# ======================================================================
# DUMMY INPUT
# ======================================================================

dummy_input = torch.randn(
    1,
    3,
    INPUT_SIZE,
    INPUT_SIZE,
    device=DEVICE
)


# ======================================================================
# INSPECT YOLOv8n INTERNAL LAYERS
# ======================================================================

print()
print("=" * 70)
print("INSPECTING YOLOv8n INTERNAL LAYERS")
print("=" * 70)

teacher_layers = teacher.model

print("Number of YOLO layers:", len(teacher_layers))

for i, layer in enumerate(teacher_layers):

    print(
        f"[{i:02d}] "
        f"{layer.__class__.__name__}"
    )


# ======================================================================
# FEATURE COLLECTION
# ======================================================================

feature_records = []


def collect_feature_hook(layer_index):

    def hook(module, inputs, output):

        # --------------------------------------------------------------
        # Some YOLO modules can return tuples/lists.
        # We only keep actual 4-D feature maps.
        # --------------------------------------------------------------

        outputs = []

        if torch.is_tensor(output):
            outputs.append(output)

        elif isinstance(output, (list, tuple)):

            for item in output:

                if torch.is_tensor(item):
                    outputs.append(item)

        # --------------------------------------------------------------
        # Store every 4-D feature map.
        # --------------------------------------------------------------

        for tensor in outputs:

            if tensor.ndim == 4:

                feature_records.append(
                    {
                        "layer_index": layer_index,
                        "layer_type": module.__class__.__name__,
                        "tensor": tensor,
                    }
                )

    return hook


# ======================================================================
# REGISTER HOOKS
# ======================================================================

hooks = []

for index, layer in enumerate(teacher_layers):

    hooks.append(
        layer.register_forward_hook(
            collect_feature_hook(index)
        )
    )


# ======================================================================
# TEACHER FORWARD PASS
# ======================================================================

print()
print("=" * 70)
print("RUNNING TEACHER FORWARD PASS")
print("=" * 70)

feature_records.clear()

with torch.no_grad():

    _ = teacher(dummy_input)


# ======================================================================
# REMOVE HOOKS
# ======================================================================

for hook in hooks:
    hook.remove()

hooks.clear()


print()
print(
    "[OK] Teacher forward pass completed"
)

print(
    "4-D feature maps collected:",
    len(feature_records)
)


# ======================================================================
# DISPLAY FEATURE MAPS
# ======================================================================

print()
print("=" * 70)
print("TEACHER FEATURE MAP CANDIDATES")
print("=" * 70)

for i, record in enumerate(feature_records):

    tensor = record["tensor"]

    print(
        f"[{i:03d}] "
        f"Layer={record['layer_index']:02d} | "
        f"{record['layer_type']:20s} | "
        f"Shape={tuple(tensor.shape)}"
    )


# ======================================================================
# FIND P3 / P4 / P5
# ======================================================================

print()
print("=" * 70)
print("SEARCHING FOR TEACHER P3 / P4 / P5")
print("=" * 70)


def find_feature_by_resolution(records, height, width):
    """
    Find the last suitable feature map having the requested
    spatial resolution.

    We search from the end because the later feature maps normally
    correspond more closely to the detector neck outputs.
    """

    candidates = []

    for record in records:

        tensor = record["tensor"]

        if (
            tensor.shape[2] == height
            and tensor.shape[3] == width
        ):
            candidates.append(record)

    if len(candidates) == 0:
        return None

    return candidates[-1]


teacher_features = {}

for level, resolution in EXPECTED_RESOLUTIONS.items():

    h, w = resolution

    record = find_feature_by_resolution(
        feature_records,
        h,
        w
    )

    if record is None:

        raise RuntimeError(
            f"\nCould not find teacher feature for {level} "
            f"with resolution {h}x{w}.\n"
            f"Please inspect the feature-map list above."
        )

    teacher_features[level] = record

    tensor = record["tensor"]

    print(
        f"[OK] {level}: "
        f"Layer {record['layer_index']} | "
        f"{record['layer_type']} | "
        f"Shape={tuple(tensor.shape)}"
    )


# ======================================================================
# TEACHER FEATURE SUMMARY
# ======================================================================

print()
print("=" * 70)
print("TEACHER FEATURE SUMMARY")
print("=" * 70)

for level in ["P3", "P4", "P5"]:

    tensor = teacher_features[level]["tensor"]

    print(
        f"{level}: "
        f"Layer={teacher_features[level]['layer_index']} | "
        f"Channels={tensor.shape[1]} | "
        f"Resolution={tensor.shape[2]}x{tensor.shape[3]}"
    )


# ======================================================================
# FEATURE ALIGNMENT MODULE
# ======================================================================

print()
print("=" * 70)
print("CREATING FEATURE ALIGNMENT MODULES")
print("=" * 70)


class FeatureAlign(nn.Module):
    """
    Lightweight 1x1 convolution used to transform
    teacher feature channels into student feature channels.

    Teacher:
        C_teacher

    Alignment:
        1x1 Conv

    Student:
        C_student
    """

    def __init__(self, teacher_channels, student_channels):

        super().__init__()

        self.projection = nn.Sequential(

            nn.Conv2d(
                teacher_channels,
                student_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False
            ),

            nn.BatchNorm2d(
                student_channels
            )

        )

    def forward(self, x):

        return self.projection(x)


# ======================================================================
# DETERMINE TEACHER CHANNELS
# ======================================================================

TEACHER_CHANNELS = {}

for level in ["P3", "P4", "P5"]:

    TEACHER_CHANNELS[level] = (
        teacher_features[level]["tensor"].shape[1]
    )

print()

for level in ["P3", "P4", "P5"]:

    print(
        f"{level}: "
        f"Teacher={TEACHER_CHANNELS[level]} -> "
        f"Student={STUDENT_CHANNELS[level]}"
    )


# ======================================================================
# CREATE ALIGNMENT NETWORK
# ======================================================================

alignment = nn.ModuleDict({

    "P3": FeatureAlign(
        TEACHER_CHANNELS["P3"],
        STUDENT_CHANNELS["P3"]
    ),

    "P4": FeatureAlign(
        TEACHER_CHANNELS["P4"],
        STUDENT_CHANNELS["P4"]
    ),

    "P5": FeatureAlign(
        TEACHER_CHANNELS["P5"],
        STUDENT_CHANNELS["P5"]
    ),

}).to(DEVICE)


# ======================================================================
# ALIGNMENT FORWARD TEST
# ======================================================================

print()
print("=" * 70)
print("TESTING FEATURE ALIGNMENT")
print("=" * 70)

alignment.eval()

aligned_features = {}

with torch.no_grad():

    for level in ["P3", "P4", "P5"]:

        teacher_tensor = (
            teacher_features[level]["tensor"]
        )

        aligned = alignment[level](
            teacher_tensor
        )

        aligned_features[level] = aligned

        print(
            f"[OK] {level}: "
            f"Teacher {tuple(teacher_tensor.shape)} "
            f"-> "
            f"Aligned {tuple(aligned.shape)}"
        )


# ======================================================================
# STUDENT SHAPE COMPATIBILITY
# ======================================================================

print()
print("=" * 70)
print("STUDENT / TEACHER COMPATIBILITY CHECK")
print("=" * 70)

expected_student_shapes = {

    "P3": (
        1,
        STUDENT_CHANNELS["P3"],
        80,
        80
    ),

    "P4": (
        1,
        STUDENT_CHANNELS["P4"],
        40,
        40
    ),

    "P5": (
        1,
        STUDENT_CHANNELS["P5"],
        20,
        20
    ),
}


for level in ["P3", "P4", "P5"]:

    aligned_shape = tuple(
        aligned_features[level].shape
    )

    expected_shape = expected_student_shapes[level]

    if aligned_shape != expected_shape:

        raise RuntimeError(
            f"{level} alignment mismatch.\n"
            f"Expected: {expected_shape}\n"
            f"Got     : {aligned_shape}"
        )

    print(
        f"[OK] {level}: "
        f"{aligned_shape}"
    )


# ======================================================================
# FEATURE DISTILLATION LOSS TEST
# ======================================================================

print()
print("=" * 70)
print("TESTING FEATURE DISTILLATION LOSS")
print("=" * 70)


def feature_distillation_loss(
    student_feature,
    aligned_teacher_feature
):
    """
    Normalized feature-level KD loss.

    Student and aligned teacher must have identical
    dimensions.
    """

    if student_feature.shape != aligned_teacher_feature.shape:

        raise RuntimeError(
            "Student and teacher feature shapes do not match:\n"
            f"Student : {student_feature.shape}\n"
            f"Teacher : {aligned_teacher_feature.shape}"
        )

    # Normalize feature vectors across channel dimension
    student_norm = F.normalize(
        student_feature,
        p=2,
        dim=1
    )

    teacher_norm = F.normalize(
        aligned_teacher_feature,
        p=2,
        dim=1
    )

    loss = F.mse_loss(
        student_norm,
        teacher_norm
    )

    return loss


# ======================================================================
# CREATE TEMPORARY STUDENT FEATURES
# ======================================================================

student_test_features = {

    "P3": torch.randn(
        1,
        STUDENT_CHANNELS["P3"],
        80,
        80,
        device=DEVICE
    ),

    "P4": torch.randn(
        1,
        STUDENT_CHANNELS["P4"],
        40,
        40,
        device=DEVICE
    ),

    "P5": torch.randn(
        1,
        STUDENT_CHANNELS["P5"],
        20,
        20,
        device=DEVICE
    ),
}


# ======================================================================
# CALCULATE TEST KD LOSS
# ======================================================================

total_kd_loss = torch.tensor(
    0.0,
    device=DEVICE
)

individual_losses = {}

for level in ["P3", "P4", "P5"]:

    loss = feature_distillation_loss(
        student_test_features[level],
        aligned_features[level]
    )

    individual_losses[level] = float(
        loss.detach().cpu().item()
    )

    total_kd_loss = total_kd_loss + loss

    print(
        f"{level} KD loss: "
        f"{individual_losses[level]:.8f}"
    )


print(
    f"\nTotal KD loss: "
    f"{float(total_kd_loss.cpu().item()):.8f}"
)


# ======================================================================
# ALIGNMENT PARAMETER COUNT
# ======================================================================

alignment_parameters = sum(
    p.numel()
    for p in alignment.parameters()
)

alignment_size_mb = (
    alignment_parameters * 4
) / (
    1024 ** 2
)


# ======================================================================
# SAVE SECTION 3 INFORMATION
# ======================================================================

output = {

    "section": 3,

    "title": (
        "Teacher / Student Feature Alignment"
    ),

    "device": str(DEVICE),

    "teacher": {

        "path": str(TEACHER_PATH),

        "architecture": "YOLOv8n",

        "input_size": INPUT_SIZE,

        "features": {

            level: {

                "layer_index":
                    int(
                        teacher_features[level]
                        ["layer_index"]
                    ),

                "layer_type":
                    teacher_features[level]
                    ["layer_type"],

                "channels":
                    int(
                        teacher_features[level]
                        ["tensor"].shape[1]
                    ),

                "height":
                    int(
                        teacher_features[level]
                        ["tensor"].shape[2]
                    ),

                "width":
                    int(
                        teacher_features[level]
                        ["tensor"].shape[3]
                    ),

            }

            for level in ["P3", "P4", "P5"]
        }
    },

    "student": {

        "architecture":
            "EcoBotX_Tiny_KD",

        "features":
            STUDENT_CHANNELS
    },

    "alignment": {

        "P3":
            f"{TEACHER_CHANNELS['P3']} -> "
            f"{STUDENT_CHANNELS['P3']}",

        "P4":
            f"{TEACHER_CHANNELS['P4']} -> "
            f"{STUDENT_CHANNELS['P4']}",

        "P5":
            f"{TEACHER_CHANNELS['P5']} -> "
            f"{STUDENT_CHANNELS['P5']}",
    },

    "alignment_parameters":
        alignment_parameters,

    "alignment_size_mb":
        alignment_size_mb,

    "test_kd_loss":
        float(
            total_kd_loss.cpu().item()
        ),

    "individual_kd_loss":
        individual_losses,

    "status":
        "PASSED"
}


JSON_PATH = (
    OUTPUT_DIR /
    "section3_feature_alignment.json"
)


with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2
    )


# ======================================================================
# FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("SECTION 3 — FINAL SUMMARY")
print("=" * 70)

print()

for level in ["P3", "P4", "P5"]:

    teacher_shape = tuple(
        teacher_features[level]["tensor"].shape
    )

    aligned_shape = tuple(
        aligned_features[level].shape
    )

    print(
        f"{level}: "
        f"Teacher {teacher_shape} "
        f"-> "
        f"Aligned {aligned_shape}"
    )

print()

print(
    "Teacher architecture : YOLOv8n"
)

print(
    "Student architecture : EcoBotX_Tiny_KD"
)

print(
    "Alignment parameters :",
    f"{alignment_parameters:,}"
)

print(
    "Alignment size        :",
    f"{alignment_size_mb:.4f} MB"
)

print(
    "KD loss test          :",
    f"{float(total_kd_loss.cpu().item()):.8f}"
)

print()

print(
    "[OK] P3 feature alignment validated."
)

print(
    "[OK] P4 feature alignment validated."
)

print(
    "[OK] P5 feature alignment validated."
)

print(
    "[OK] Feature KD loss validated."
)

print()

print(
    "Results saved to:"
)

print(
    JSON_PATH
)

print()

print(
    "[OK] SECTION 3 COMPLETED."
)

print(
    "Ready for SECTION 4 — Knowledge Distillation Training."
)

print("=" * 70)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 3 — TEACHER / STUDENT FEATURE ALIGNMENT

Device       : cuda
Teacher      : G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt
Input size   : 640x640
Student P3   : 64
Student P4   : 96
Student P5   : 128

LOADING YOLOv8n TEACHER
[OK] Teacher loaded
[OK] Teacher frozen
Teacher type: DetectionModel

INSPECTING YOLOv8n INTERNAL LAYERS
Number of YOLO layers: 23
[00] Conv
[01] Conv
[02] C2f
[03] Conv
[04] C2f
[05] Conv
[06] C2f
[07] Conv
[08] C2f
[09] SPPF
[10] Upsample
[11] Concat
[12] C2f
[13] Upsample
[14] Concat
[15] C2f
[16] Conv
[17] Concat
[18] C2f
[19] Conv
[20] Concat
[21] C2f
[22] Detect

RUNNING TEACHER FORWARD PASS

[OK] Teacher forward pass completed
4-D feature maps collected: 22

TEACHER FEATURE MAP CANDIDATES
[000] Layer=00 | Conv                 | Shape=(1, 16, 320, 320)
[001] Layer=01 | Conv                 | Shape=(1, 32, 160, 160)
[002] Layer=02 | C2f                  | Shape=(1, 32, 160, 160)
[003] Layer=03 | Conv          

In [ ]:

# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 4 — KNOWLEDGE DISTILLATION TRAINING
# CORRECTED / STABLE VERSION
# ======================================================================

import os
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from ultralytics import YOLO


print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 4 — KNOWLEDGE DISTILLATION TRAINING")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASET_ROOT = Path(
    r"G:\EcoBotX_YOLO"
)

TRAIN_IMAGES = DATASET_ROOT / "images" / "train"
TRAIN_LABELS = DATASET_ROOT / "labels" / "train"

VAL_IMAGES = DATASET_ROOT / "images" / "val"
VAL_LABELS = DATASET_ROOT / "labels" / "val"

DATASET_YAML = DATASET_ROOT / "dataset.yaml"


# ======================================================================
# TEACHER
# ======================================================================

TEACHER_PATH = Path(
    r"G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt"
)


# ======================================================================
# OUTPUT
# ======================================================================

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\knowledge_distillation"
    r"\section4_kd_training"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# TRAINING CONFIGURATION
# ======================================================================

IMAGE_SIZE = 640

BATCH_SIZE = 8

EPOCHS = 100

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 5e-4

NUM_WORKERS = 4

KD_WEIGHT = 1.0

GRAD_CLIP = 10.0

SAVE_PERIOD = 10

NUM_CLASSES = 4

CLASS_NAMES = [
    "BOTTLE",
    "CAN",
    "PAPER",
    "WRAPPER"
]


# ======================================================================
# STUDENT FEATURE CHANNELS
# ======================================================================

STUDENT_CHANNELS = {
    "P3": 64,
    "P4": 96,
    "P5": 128
}


EXPECTED_RESOLUTIONS = {
    "P3": (80, 80),
    "P4": (40, 40),
    "P5": (20, 20)
}


# ======================================================================
# PRINT CONFIGURATION
# ======================================================================

print()
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)

print("Device        :", DEVICE)
print("Dataset       :", DATASET_ROOT)
print("Train images  :", TRAIN_IMAGES)
print("Train labels  :", TRAIN_LABELS)
print("Val images    :", VAL_IMAGES)
print("Val labels    :", VAL_LABELS)
print("Teacher       :", TEACHER_PATH)
print("Image size    :", IMAGE_SIZE)
print("Batch size    :", BATCH_SIZE)
print("Epochs        :", EPOCHS)
print("Learning rate :", LEARNING_RATE)
print("KD weight     :", KD_WEIGHT)

print()


# ======================================================================
# 2. PATH VALIDATION
# ======================================================================

print("=" * 70)
print("PATH VALIDATION")
print("=" * 70)

required_paths = {
    "Dataset YAML": DATASET_YAML,
    "Train images": TRAIN_IMAGES,
    "Train labels": TRAIN_LABELS,
    "Validation images": VAL_IMAGES,
    "Validation labels": VAL_LABELS,
    "Teacher model": TEACHER_PATH
}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"\n{name} not found:\n{path}"
        )

    print(f"[OK] {name}: {path}")


# ======================================================================
# 3. VERIFY SECTION 2 / SECTION 3 OBJECTS
# ======================================================================

print()
print("=" * 70)
print("CHECKING PREVIOUS SECTIONS")
print("=" * 70)


# IMPORTANT:
# Section 2 defines:
#
#     class EcoBotXStudent(nn.Module)
#
# NOT:
#
#     class EcoBotX_Tiny_KD
#
# Therefore we correctly check EcoBotXStudent here.

if "EcoBotXStudent" not in globals():

    raise RuntimeError(
        "\nEcoBotXStudent was not found.\n"
        "Please run SECTION 2 successfully before SECTION 4."
    )


if "FeatureAlign" not in globals():

    raise RuntimeError(
        "\nFeatureAlign was not found.\n"
        "Please run SECTION 3 successfully before SECTION 4."
    )


print("[OK] EcoBotXStudent found")
print("[OK] FeatureAlign found")


# ======================================================================
# 4. LOAD YOLOv8n TEACHER
# ======================================================================

print()
print("=" * 70)
print("LOADING YOLOv8n TEACHER")
print("=" * 70)


teacher_wrapper = YOLO(
    str(TEACHER_PATH)
)

teacher = teacher_wrapper.model

teacher = teacher.to(DEVICE)

teacher.eval()


# Freeze teacher completely

for parameter in teacher.parameters():

    parameter.requires_grad = False


print("[OK] Teacher loaded")
print("[OK] Teacher frozen")
print("Teacher type :", type(teacher).__name__)


# ======================================================================
# 5. CREATE STUDENT
# ======================================================================

print()
print("=" * 70)
print("CREATING STUDENT")
print("=" * 70)


# CORRECT CLASS NAME FROM SECTION 2
student = EcoBotXStudent(
    num_classes=NUM_CLASSES
).to(DEVICE)


print("[OK] Student created")


# ======================================================================
# 6. STUDENT PARAMETER COUNT
# ======================================================================

student_parameters = sum(
    p.numel()
    for p in student.parameters()
)

trainable_student_parameters = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)

student_size_mb = (
    student_parameters * 4
) / (
    1024 ** 2
)


print()
print("Student parameters          :", f"{student_parameters:,}")
print("Trainable student params    :", f"{trainable_student_parameters:,}")
print("Student FP32 size           :", f"{student_size_mb:.3f} MB")


# ======================================================================
# 7. DETERMINE TEACHER FEATURE CHANNELS
# ======================================================================

print()
print("=" * 70)
print("IDENTIFYING TEACHER P3 / P4 / P5")
print("=" * 70)


teacher_feature_records = []


def collect_teacher_features(
    module,
    inputs,
    output
):

    tensors = []

    if torch.is_tensor(output):

        tensors.append(output)

    elif isinstance(output, (list, tuple)):

        for item in output:

            if torch.is_tensor(item):

                tensors.append(item)

    for tensor in tensors:

        if tensor.ndim == 4:

            teacher_feature_records.append(
                {
                    "layer_index": None,
                    "layer_type": module.__class__.__name__,
                    "tensor": tensor
                }
            )


# ----------------------------------------------------------------------
# Register hooks
# ----------------------------------------------------------------------

teacher_hooks = []

for layer in teacher.model:

    teacher_hooks.append(
        layer.register_forward_hook(
            collect_teacher_features
        )
    )


# ----------------------------------------------------------------------
# Dummy forward
# ----------------------------------------------------------------------

dummy_input = torch.zeros(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE
)


teacher_feature_records.clear()


with torch.no_grad():

    _ = teacher(
        dummy_input
    )


# ----------------------------------------------------------------------
# Remove hooks
# ----------------------------------------------------------------------

for hook in teacher_hooks:

    hook.remove()

teacher_hooks.clear()


print(
    "[OK] Teacher feature inspection completed."
)

print(
    "4-D feature maps found:",
    len(teacher_feature_records)
)


# ======================================================================
# 8. FEATURE SELECTION
# ======================================================================

def select_teacher_feature(
    records,
    height,
    width
):

    candidates = []

    for record in records:

        tensor = record["tensor"]

        if (
            tensor.shape[2] == height
            and tensor.shape[3] == width
        ):

            candidates.append(record)

    if len(candidates) == 0:

        raise RuntimeError(
            f"No teacher feature found for "
            f"resolution {height}x{width}."
        )

    # Last feature at this resolution
    return candidates[-1]


teacher_features_info = {

    "P3": select_teacher_feature(
        teacher_feature_records,
        80,
        80
    ),

    "P4": select_teacher_feature(
        teacher_feature_records,
        40,
        40
    ),

    "P5": select_teacher_feature(
        teacher_feature_records,
        20,
        20
    )
}


for level in ["P3", "P4", "P5"]:

    tensor = teacher_features_info[level]["tensor"]

    print(
        f"{level}: "
        f"Shape={tuple(tensor.shape)}"
    )


# ======================================================================
# 9. TEACHER CHANNEL INFORMATION
# ======================================================================

TEACHER_CHANNELS = {

    level:
        int(
            teacher_features_info[level]["tensor"].shape[1]
        )

    for level in ["P3", "P4", "P5"]
}


print()
print("=" * 70)
print("TEACHER / STUDENT CHANNEL ALIGNMENT")
print("=" * 70)


for level in ["P3", "P4", "P5"]:

    print(
        f"{level}: "
        f"Teacher={TEACHER_CHANNELS[level]} "
        f"-> "
        f"Student={STUDENT_CHANNELS[level]}"
    )


# ======================================================================
# 10. CREATE FEATURE ALIGNMENT NETWORK
# ======================================================================

alignment = nn.ModuleDict({

    "P3": FeatureAlign(
        TEACHER_CHANNELS["P3"],
        STUDENT_CHANNELS["P3"]
    ),

    "P4": FeatureAlign(
        TEACHER_CHANNELS["P4"],
        STUDENT_CHANNELS["P4"]
    ),

    "P5": FeatureAlign(
        TEACHER_CHANNELS["P5"],
        STUDENT_CHANNELS["P5"]
    )

}).to(DEVICE)


print()
print("[OK] Feature alignment network created.")


# ======================================================================
# 11. VERIFY STUDENT FORWARD PASS
# ======================================================================

print()
print("=" * 70)
print("VERIFYING STUDENT")
print("=" * 70)


student.eval()


with torch.no_grad():

    student_test_output = student(
        dummy_input
    )


student_test_features = (
    student_test_output["P3"],
    student_test_output["P4"],
    student_test_output["P5"]
)


for level, feature in zip(
    ["P3", "P4", "P5"],
    student_test_features
):

    print(
        f"{level}: "
        f"{tuple(feature.shape)}"
    )


# ======================================================================
# 12. FEATURE KD LOSS
# ======================================================================

def feature_kd_loss(
    student_feature,
    teacher_feature
):

    # --------------------------------------------------------------
    # Verify dimensions
    # --------------------------------------------------------------

    if student_feature.shape != teacher_feature.shape:

        raise RuntimeError(
            "\nFeature shape mismatch.\n"
            f"Student : {tuple(student_feature.shape)}\n"
            f"Teacher : {tuple(teacher_feature.shape)}"
        )


    # --------------------------------------------------------------
    # L2 normalization across channel dimension
    # --------------------------------------------------------------

    student_norm = F.normalize(
        student_feature,
        p=2,
        dim=1
    )

    teacher_norm = F.normalize(
        teacher_feature,
        p=2,
        dim=1
    )


    # --------------------------------------------------------------
    # MSE feature distillation
    # --------------------------------------------------------------

    return F.mse_loss(
        student_norm,
        teacher_norm
    )


# ======================================================================
# 13. DATASET
# ======================================================================

class YOLODetectionDataset(Dataset):

    def __init__(
        self,
        image_dir,
        label_dir,
        image_size=640
    ):

        self.image_dir = Path(image_dir)

        self.label_dir = Path(label_dir)

        self.image_size = image_size


        valid_extensions = {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        }


        self.images = sorted(
            [
                p
                for p in self.image_dir.iterdir()
                if p.suffix.lower()
                in valid_extensions
            ]
        )


        if len(self.images) == 0:

            raise RuntimeError(
                f"No images found in:\n"
                f"{self.image_dir}"
            )


    def __len__(self):

        return len(self.images)


    def __getitem__(
        self,
        index
    ):

        image_path = self.images[index]

        label_path = (
            self.label_dir /
            f"{image_path.stem}.txt"
        )


        # ----------------------------------------------------------
        # Load image
        # ----------------------------------------------------------

        image = Image.open(
            image_path
        ).convert("RGB")


        image = image.resize(
            (
                self.image_size,
                self.image_size
            )
        )


        image_array = np.array(
            image,
            dtype=np.float32
        )


        image_tensor = torch.from_numpy(
            image_array
        ).permute(
            2,
            0,
            1
        ) / 255.0


        # ----------------------------------------------------------
        # Read labels
        # ----------------------------------------------------------

        targets = []


        if label_path.exists():

            with open(
                label_path,
                "r",
                encoding="utf-8"
            ) as f:

                for line in f:

                    parts = line.strip().split()

                    if len(parts) != 5:

                        continue


                    try:

                        class_id = int(parts[0])

                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])

                    except ValueError:

                        continue


                    targets.append(
                        [
                            class_id,
                            xc,
                            yc,
                            w,
                            h
                        ]
                    )


        targets = torch.tensor(
            targets,
            dtype=torch.float32
        )


        return (
            image_tensor,
            targets,
            str(image_path)
        )


# ======================================================================
# 14. COLLATE FUNCTION
# ======================================================================

def detection_collate(batch):

    images = torch.stack(
        [
            item[0]
            for item in batch
        ]
    )


    targets = [
        item[1]
        for item in batch
    ]


    paths = [
        item[2]
        for item in batch
    ]


    return (
        images,
        targets,
        paths
    )


# ======================================================================
# 15. CREATE DATASETS
# ======================================================================

print()
print("=" * 70)
print("CREATING DATASETS")
print("=" * 70)


train_dataset = YOLODetectionDataset(
    TRAIN_IMAGES,
    TRAIN_LABELS,
    IMAGE_SIZE
)


val_dataset = YOLODetectionDataset(
    VAL_IMAGES,
    VAL_LABELS,
    IMAGE_SIZE
)


print(
    "Training images   :",
    len(train_dataset)
)


print(
    "Validation images :",
    len(val_dataset)
)


# ======================================================================
# 16. CREATE DATALOADERS
# ======================================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),

    collate_fn=detection_collate,

    drop_last=False
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),

    collate_fn=detection_collate,

    drop_last=False
)


print("[OK] DataLoaders created.")


# ======================================================================
# 17. OPTIMIZER
# ======================================================================

trainable_parameters = list(
    student.parameters()
) + list(
    alignment.parameters()
)


optimizer = torch.optim.AdamW(

    trainable_parameters,

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


# ======================================================================
# 18. LEARNING-RATE SCHEDULER
# ======================================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS,

    eta_min=LEARNING_RATE * 0.01
)


print()
print("[OK] Optimizer created.")
print("[OK] Scheduler created.")


# ======================================================================
# 19. TRAINING HISTORY
# ======================================================================

history = {

    "epoch": [],

    "train_kd_loss": [],

    "val_kd_loss": [],

    "learning_rate": [],

    "epoch_time": []
}


# ======================================================================
# 20. CHECKPOINT FUNCTION
# ======================================================================

def save_checkpoint(
    epoch,
    filename
):

    checkpoint = {

        "epoch":
            epoch,

        "student_state_dict":
            student.state_dict(),

        "alignment_state_dict":
            alignment.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "student_parameters":
            student_parameters,

        "teacher_path":
            str(TEACHER_PATH),

        "teacher_channels":
            TEACHER_CHANNELS,

        "student_channels":
            STUDENT_CHANNELS,

        "kd_weight":
            KD_WEIGHT,

        "image_size":
            IMAGE_SIZE,

        "history":
            history
    }


    path = OUTPUT_DIR / filename


    torch.save(
        checkpoint,
        path
    )


    return path


# ======================================================================
# 21. TRAINING
# ======================================================================

print()
print("=" * 70)
print("STARTING FEATURE KNOWLEDGE DISTILLATION TRAINING")
print("=" * 70)

print()


best_val_loss = float("inf")

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()


    # ==============================================================
    # TRAIN MODE
    # ==============================================================

    student.train()

    alignment.train()


    total_train_loss = 0.0

    train_batches = 0


    # ==============================================================
    # TRAIN BATCHES
    # ==============================================================

    for batch_index, (
        images,
        targets,
        paths
    ) in enumerate(train_loader, start=1):


        images = images.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------------
        # STUDENT FORWARD
        # ----------------------------------------------------------

        student_output = student(
            images
        )


        student_p3 = student_output["P3"]
        student_p4 = student_output["P4"]
        student_p5 = student_output["P5"]


        # ----------------------------------------------------------
        # TEACHER FEATURE COLLECTION
        # ----------------------------------------------------------

        teacher_feature_records = []


        def batch_teacher_hook(
            module,
            inputs,
            output
        ):

            tensors = []


            if torch.is_tensor(output):

                tensors.append(output)


            elif isinstance(
                output,
                (list, tuple)
            ):

                for item in output:

                    if torch.is_tensor(item):

                        tensors.append(item)


            for tensor in tensors:

                if tensor.ndim == 4:

                    teacher_feature_records.append(
                        {
                            "tensor": tensor
                        }
                    )


        teacher_hooks = []


        for layer in teacher.model:

            teacher_hooks.append(
                layer.register_forward_hook(
                    batch_teacher_hook
                )
            )


        # ----------------------------------------------------------
        # TEACHER FORWARD
        # ----------------------------------------------------------

        with torch.no_grad():

            _ = teacher(
                images
            )


        # ----------------------------------------------------------
        # REMOVE HOOKS IMMEDIATELY
        # ----------------------------------------------------------

        for hook in teacher_hooks:

            hook.remove()


        teacher_hooks.clear()


        # ----------------------------------------------------------
        # FIND TEACHER FEATURES
        # ----------------------------------------------------------

        teacher_p3 = select_teacher_feature(
            teacher_feature_records,
            80,
            80
        )["tensor"]


        teacher_p4 = select_teacher_feature(
            teacher_feature_records,
            40,
            40
        )["tensor"]


        teacher_p5 = select_teacher_feature(
            teacher_feature_records,
            20,
            20
        )["tensor"]


        # ----------------------------------------------------------
        # ALIGN TEACHER FEATURES
        # ----------------------------------------------------------

        aligned_p3 = alignment["P3"](
            teacher_p3
        )


        aligned_p4 = alignment["P4"](
            teacher_p4
        )


        aligned_p5 = alignment["P5"](
            teacher_p5
        )


        # ----------------------------------------------------------
        # KD LOSSES
        # ----------------------------------------------------------

        loss_p3 = feature_kd_loss(
            student_p3,
            aligned_p3
        )


        loss_p4 = feature_kd_loss(
            student_p4,
            aligned_p4
        )


        loss_p5 = feature_kd_loss(
            student_p5,
            aligned_p5
        )


        kd_loss = (
            loss_p3 +
            loss_p4 +
            loss_p5
        ) / 3.0


        total_loss = (
            KD_WEIGHT *
            kd_loss
        )


        # ----------------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------------

        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            GRAD_CLIP
        )


        optimizer.step()


        total_train_loss += (
            total_loss.detach().item()
        )


        train_batches += 1


    # ==============================================================
    # VALIDATION
    # ==============================================================

    student.eval()

    alignment.eval()


    total_val_loss = 0.0

    val_batches = 0


    with torch.no_grad():

        for (
            images,
            targets,
            paths
        ) in val_loader:


            images = images.to(
                DEVICE,
                non_blocking=True
            )


            # ------------------------------------------------------
            # STUDENT
            # ------------------------------------------------------

            student_output = student(
                images
            )


            student_p3 = student_output["P3"]
            student_p4 = student_output["P4"]
            student_p5 = student_output["P5"]


            # ------------------------------------------------------
            # TEACHER
            # ------------------------------------------------------

            teacher_feature_records = []


            def val_teacher_hook(
                module,
                inputs,
                output
            ):

                tensors = []


                if torch.is_tensor(output):

                    tensors.append(output)


                elif isinstance(
                    output,
                    (list, tuple)
                ):

                    for item in output:

                        if torch.is_tensor(item):

                            tensors.append(item)


                for tensor in tensors:

                    if tensor.ndim == 4:

                        teacher_feature_records.append(
                            {
                                "tensor": tensor
                            }
                        )


            teacher_hooks = []


            for layer in teacher.model:

                teacher_hooks.append(
                    layer.register_forward_hook(
                        val_teacher_hook
                    )
                )


            _ = teacher(
                images
            )


            for hook in teacher_hooks:

                hook.remove()


            teacher_hooks.clear()


            # ------------------------------------------------------
            # TEACHER FEATURES
            # ------------------------------------------------------

            teacher_p3 = select_teacher_feature(
                teacher_feature_records,
                80,
                80
            )["tensor"]


            teacher_p4 = select_teacher_feature(
                teacher_feature_records,
                40,
                40
            )["tensor"]


            teacher_p5 = select_teacher_feature(
                teacher_feature_records,
                20,
                20
            )["tensor"]


            # ------------------------------------------------------
            # ALIGNMENT
            # ------------------------------------------------------

            aligned_p3 = alignment["P3"](
                teacher_p3
            )


            aligned_p4 = alignment["P4"](
                teacher_p4
            )


            aligned_p5 = alignment["P5"](
                teacher_p5
            )


            # ------------------------------------------------------
            # VALIDATION KD LOSS
            # ------------------------------------------------------

            val_p3 = feature_kd_loss(
                student_p3,
                aligned_p3
            )


            val_p4 = feature_kd_loss(
                student_p4,
                aligned_p4
            )


            val_p5 = feature_kd_loss(
                student_p5,
                aligned_p5
            )


            val_loss = (
                val_p3 +
                val_p4 +
                val_p5
            ) / 3.0


            total_val_loss += (
                val_loss.item()
            )


            val_batches += 1


    # ==============================================================
    # EPOCH RESULTS
    # ==============================================================

    avg_train_loss = (
        total_train_loss /
        max(train_batches, 1)
    )


    avg_val_loss = (
        total_val_loss /
        max(val_batches, 1)
    )


    scheduler.step()


    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    epoch_time = (
        time.time() -
        epoch_start
    )


    history["epoch"].append(
        epoch
    )


    history["train_kd_loss"].append(
        avg_train_loss
    )


    history["val_kd_loss"].append(
        avg_val_loss
    )


    history["learning_rate"].append(
        current_lr
    )


    history["epoch_time"].append(
        epoch_time
    )


    # ==============================================================
    # PRINT
    # ==============================================================

    print(
        f"Epoch "
        f"{epoch:03d}/{EPOCHS} | "
        f"Train KD: {avg_train_loss:.6f} | "
        f"Val KD: {avg_val_loss:.6f} | "
        f"LR: {current_lr:.7f} | "
        f"Time: {epoch_time:.1f}s"
    )


    # ==============================================================
    # BEST CHECKPOINT
    # ==============================================================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss


        best_path = save_checkpoint(
            epoch,
            "best_student_kd.pt"
        )


        print(
            f"   [BEST] "
            f"Validation KD loss: "
            f"{best_val_loss:.6f}"
        )


    # ==============================================================
    # PERIODIC CHECKPOINT
    # ==============================================================

    if (
        epoch % SAVE_PERIOD == 0
        or epoch == EPOCHS
    ):

        checkpoint_path = save_checkpoint(
            epoch,
            f"student_kd_epoch_{epoch:03d}.pt"
        )


        print(
            f"   [CHECKPOINT] "
            f"{checkpoint_path}"
        )


# ======================================================================
# 22. TRAINING COMPLETED
# ======================================================================

total_training_time = (
    time.time() -
    training_start
)


print()
print("=" * 70)
print("SECTION 4 — TRAINING COMPLETED")
print("=" * 70)

print()

print(
    "Total epochs       :",
    EPOCHS
)

print(
    "Best validation KD :",
    f"{best_val_loss:.6f}"
)

print(
    "Training time      :",
    f"{total_training_time / 3600:.2f} hours"
)

print(
    "Student parameters :",
    f"{student_parameters:,}"
)

print(
    "Student FP32 size  :",
    f"{student_size_mb:.3f} MB"
)


# ======================================================================
# 23. SAVE TRAINING HISTORY
# ======================================================================

history_path = (
    OUTPUT_DIR /
    "section4_training_history.json"
)


with open(
    history_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


print()
print(
    "[OK] Training history saved:"
)

print(
    history_path
)


# ======================================================================
# 24. SAVE FINAL STUDENT
# ======================================================================

final_student_path = (
    OUTPUT_DIR /
    "final_student_kd.pt"
)


torch.save(
    {
        "student_state_dict":
            student.state_dict(),

        "alignment_state_dict":
            alignment.state_dict(),

        "student_parameters":
            student_parameters,

        "teacher_path":
            str(TEACHER_PATH),

        "teacher_channels":
            TEACHER_CHANNELS,

        "student_channels":
            STUDENT_CHANNELS,

        "best_val_kd_loss":
            best_val_loss,

        "image_size":
            IMAGE_SIZE,

        "class_names":
            CLASS_NAMES
    },
    final_student_path
)


print(
    "[OK] Final student checkpoint saved:"
)

print(
    final_student_path
)


# ======================================================================
# 25. FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("SECTION 4 — FINAL SUMMARY")
print("=" * 70)

print()

print(
    "Teacher architecture : YOLOv8n"
)

print(
    "Student architecture : EcoBotXStudent"
)

print(
    "KD type              : Feature-level KD"
)

print(
    "Distillation levels  : P3 / P4 / P5"
)

print(
    "P3 teacher channels   :",
    TEACHER_CHANNELS["P3"]
)

print(
    "P4 teacher channels   :",
    TEACHER_CHANNELS["P4"]
)

print(
    "P5 teacher channels   :",
    TEACHER_CHANNELS["P5"]
)

print(
    "P3 student channels  :",
    STUDENT_CHANNELS["P3"]
)

print(
    "P4 student channels  :",
    STUDENT_CHANNELS["P4"]
)

print(
    "P5 student channels  :",
    STUDENT_CHANNELS["P5"]
)

print(
    "Student parameters   :",
    f"{student_parameters:,}"
)

print(
    "Best validation KD   :",
    f"{best_val_loss:.6f}"
)

print(
    "History file         :",
    history_path
)

print(
    "Final student        :",
    final_student_path
)

print()

print("=" * 70)
print("[OK] SECTION 4 COMPLETED SUCCESSFULLY")
print("=" * 70)

print()

print(
    "Ready for SECTION 5 — "
    "Student detection-head integration."
)

print("=" * 70)


ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 4 — KNOWLEDGE DISTILLATION TRAINING

CONFIGURATION
Device        : cuda
Dataset       : G:\EcoBotX_YOLO
Train images  : G:\EcoBotX_YOLO\images\train
Train labels  : G:\EcoBotX_YOLO\labels\train
Val images    : G:\EcoBotX_YOLO\images\val
Val labels    : G:\EcoBotX_YOLO\labels\val
Teacher       : G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt
Image size    : 640
Batch size    : 8
Epochs        : 100
Learning rate : 0.001
KD weight     : 1.0

PATH VALIDATION
[OK] Dataset YAML: G:\EcoBotX_YOLO\dataset.yaml
[OK] Train images: G:\EcoBotX_YOLO\images\train
[OK] Train labels: G:\EcoBotX_YOLO\labels\train
[OK] Validation images: G:\EcoBotX_YOLO\images\val
[OK] Validation labels: G:\EcoBotX_YOLO\labels\val
[OK] Teacher model: G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt

CHECKING PREVIOUS SECTIONS
[OK] EcoBotXStudent found
[OK] FeatureAlign found

LOADING YOLOv8n TEACHER
[OK] Teacher loaded
[OK] Teacher frozen
Teacher type : Detection